<a href="https://colab.research.google.com/github/Lachlan-Walker/BE-Project/blob/main/Max_Cut.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scipy
!pip install ortools
!pip install gurobipy
!pip install dwave-ocean-sdk
!pip install dwave-preprocessing

In [1]:
import numpy as np
import os
import dimod
import time
import pandas as pd
import matplotlib.pyplot as plt
from dwave.samplers import TabuSampler
from dwave.samplers import PathIntegralAnnealingSampler, RotorModelAnnealingSampler
from dwave.samplers import SimulatedAnnealingSampler
from ortools.sat.python import cp_model
import re
import csv
import traceback
import gurobipy as gp
from gurobipy import GRB
import dwave_networkx as dnx
import minorminer
from dwave.embedding import embed_bqm, unembed_sampleset
from dwave.embedding.chain_breaks import majority_vote, discard, MinimizeEnergy
import random
import networkx as nx
from minorminer import find_embedding
from collections import deque

In [2]:
DATASET_DIR = "/content/drive/MyDrive/MaxCut_Data/ER_dataset"
RESULTS_CSV = "/content/drive/MyDrive/MaxCut_Data/Results/raw_results.csv"
PROGRESS_CSV = "/content/drive/MyDrive/MaxCut_Data/Results/progress_logs.csv"

# Make sure results folder exists
os.makedirs(os.path.dirname(RESULTS_CSV), exist_ok=True)
os.makedirs(os.path.dirname(PROGRESS_CSV), exist_ok=True)

# **Dataset Summary**
Contains 10 randomly generated ER graphs per number of nodes and expected degree. 500 instances in total, densities for each instance shown below:


In [ ]:
Exp_degree = [0.6, 0.8, 1.0, 1.6, 2, 3, 5, 8, 13, 20]
Nodes = [100, 1_000, 10_000, 100_000, 1_000_000]

df = pd.DataFrame(
    [[round(c/n, 10) for n in Nodes] for c in Exp_degree],
    index=Exp_degree,
    columns=Nodes
)

df

,100,1000,10000,100000,1000000
0.6,0.006,0.0006,0.00006,0.000006,6.000000e-07
0.8,0.008,0.0008,0.00008,0.000008,8.000000e-07
1.0,0.010,0.0010,0.00010,0.000010,1.000000e-06
1.6,0.016,0.0016,0.00016,0.000016,1.600000e-06
2.0,0.020,0.0020,0.00020,0.000020,2.000000e-06
3.0,0.030,0.0030,0.00030,0.000030,3.000000e-06
5.0,0.050,0.0050,0.00050,0.000050,5.000000e-06
8.0,0.080,0.0080,0.00080,0.000080,8.000000e-06
13.0,0.130,0.0130,0.00130,0.000130,1.300000e-05
20.0,0.200,0.0200,0.00200,0.000200,2.000000e-05


# Helper functions

In [3]:
def parse_instance_filename(filename):
    """
    Take a file in the dataset and return:
    (n_vars:int, c:float, instance_id:int)
    """
    if not filename.endswith(".txt"):
        return None

    stem = os.path.splitext(filename)[0]
    parts = stem.split("_")

    n_vars = int(parts[0])
    c = float(parts[1])
    instance_id = int(parts[2])

    return n_vars, c, instance_id

#=======================================================================
def append_progress_rows(csv_path, rows):
    if not csv_path or not rows:
        return

    fieldnames = [
        "solver",
        "instance_path",
        "n_nodes",
        "n_edges",
        "exp_degree",
        "instance_id",
        "time_s",
        "event_idx",
        "objective",
        "best_bound",
        "gap",
        "iteration",
        "note",
    ]

    file_exists = os.path.exists(csv_path)

    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)

#=============================================================================
def ensure_results_csv(csv_path: str, fieldnames: list[str]):
    """
    Create results CSV with header if it does not exist.
    """
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)

    if not os.path.exists(csv_path):
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
#=========================================================================

def load_done_keys(csv_path: str):
    if not os.path.exists(csv_path):
        return set()
    df = pd.read_csv(csv_path)
    if df.empty:
        return set()

    required = {"solver", "n_nodes", "exp_degree", "instance_id", "status"}
    if not required.issubset(df.columns):
        return set()

    ok = ~df["status"].astype(str).str.startswith("ERROR")
    df_ok = df.loc[ok]

    return set(zip(df_ok["solver"], df_ok["n_nodes"], df_ok["exp_degree"], df_ok["instance_id"]))


#=============================================================================
def collect_instance_files(instance_dir: str, node_size: int):
    """
    Collect all instance files with given node size.

    Returns a sorted list of tuples:
      (exp_degree, instance_id, full_path)
    """
    collected = []

    for fname in os.listdir(instance_dir):
        meta = parse_instance_filename(fname)
        if meta is None:
            continue

        n, d, inst = meta
        if n == node_size:
            collected.append((d, inst, os.path.join(instance_dir, fname)))

    # Sort for deterministic ordering
    collected.sort(key=lambda t: (t[0], t[1]))
    return collected

#================================================================================
def compute_gap(objective, best_bound):
    """
    Compute optimality gap for maximization problems.

    gap = (best_bound - objective) / best_bound

    Returns None if no bound is available.
    """
    if best_bound is None:
        return None

    denom = max(1.0, float(best_bound))
    return (float(best_bound) - float(objective)) / denom

#==============================================================================
def append_result_row(csv_path: str, fieldnames: list[str], row: dict):
    """
    Append a single result row to the CSV.
    """
    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow(row)
#==============================================================================
def build_result_row(
    solver: str,
    n_nodes: int,
    exp_degree: float,
    instance_id: int,
    objective: int,
    best_bound,
    wall_time: float,
    time_limit: float,
    status: str,
):
    """
    Build a standardized CSV row dict.
    """
    return {
        "solver": solver,
        "n_nodes": n_nodes,
        "exp_degree": exp_degree,
        "instance_id": instance_id,
        "objective": objective,
        "best_bound": best_bound,
        "gap": compute_gap(objective, best_bound),
        "wall_time": wall_time,
        "time_limit": time_limit,
        "status": status,
    }
#===============================================================================
def get_Q(filename):
    '''
    Creates the Q matrix for a given problem instance
    '''
    Q = {}

    with open(filename, "r") as f:

      for line in f:
          line = line.strip()

          parts = line.split()

          # read only endpoints, ignore weight if present
          i, j = map(int, parts[:2])

          w = 1  # force unit weight

          # diagonal terms
          Q[(i, i)] = Q.get((i, i), 0) - w
          Q[(j, j)] = Q.get((j, j), 0) - w

          # off-diagonal (upper triangular)
          a, b = sorted((i, j))
          Q[(a, b)] = Q.get((a, b), 0) + 2 * w

      return Q


# **Simulated Annealing sovler**



In [4]:
def SA_solve(filename,
             num_reads: int = 3,
             num_sweeps: int=500,
             beta_range=[0.1,5],
             beta_schedule_type: str = 'linear'):

    t0 = time.perf_counter() # external timing
    Q = get_Q(filename) # Create the QUBO

    bqm = dimod.BinaryQuadraticModel(Q, "BINARY")
    sa_solver = SimulatedAnnealingSampler()
    solution = sa_solver.sample(
        bqm,
        num_reads=num_reads,
        num_sweeps=num_sweeps,
        beta_range=beta_range,
        beta_schedule_type=beta_schedule_type,
        seed=37
    )

    best_energy = solution.first.energy
    objective = int(round(-best_energy)) # Report as positive for consistency

    runtime = time.perf_counter() - t0

    return {
        "objective": objective,
        "best_bound": None,
        "wall_time": runtime,
        "status": "Feasible"
    }

# **Simulated Quantum Annealing with Hardware Constraints**

## V2


In [5]:
def SQA_solve_hardware_like_v2(
    filename,
    pegasus_m=16,                 # P16 ~ 5000 qubits
    chain_strength=1.8,
    chain_break_method="minimize_energy",  # "majority_vote" / "discard" / "minimize_energy"

    # ---- SQA params ----
    num_reads=1,
    num_sweeps=500,
    beta_range=(0.1, 3.0),
    Hp_field=None,
    Hd_field=None,
    trans_fields=0.25,
    num_sweeps_per_beta=5,
    randomize_order=False,
    beta_schedule_type="geometric",
    proposal_acceptance_criteria="MetropolisNonErgodic",
    seed=37,

    # ---- decomposition controls ----
    subproblem_size=200,
    max_iters=3000,
    patience=400,
    init_method="random",
    tile_pool_size=128,
    reuse_cycles=50,
    accept_worse_prob=0.0,
    boundary_scale_sparse=0.4,
    boundary_scale_dense=0.9,

    max_internal_edges=2500,      # hard cap on internal edges per tile
    min_subproblem_size=50,       # floor so tiles don't become trivially small

    pool_refresh_every=None,      # iterations between pool refresh (None = auto)
    refresh_fraction=0.25,        # fraction of pool to retire each refresh
    coverage_bias=2.0,            # how strongly to bias seeding toward unvisited nodes

    boundary_scale_min=0.2,       # floor for per-tile boundary scale
    boundary_scale_max=1.5,       # ceiling for per-tile boundary scale

    warm_start=True,              # pass current x as initial_states hint to sampler

    cut_bias=1.5,                 # weight bonus for seeding from hot edges
    hot_edge_fraction=0.1,        # fraction of non-cut edges considered "hot"

    # ---- progress logging ----
    progress_csv_path=None,
    exp_degree=None,
    instance_id=None,
    log_checkpoints=False,
    checkpoint_every=50,

    verbose=False,
):
    """
    Hardware-like SQA Max-Cut solver with decomposition and progress logging.
    """

    overall_t0 = time.perf_counter()
    progress_rows = []
    event_idx = 0

    def log_progress(objective, iteration=None, note="improved"):
        nonlocal event_idx
        if progress_csv_path is None:
            return
        progress_rows.append({
            "solver": "SQA",
            "instance_path": filename,
            "n_nodes": int(n) if "n" in locals() else None,
            "n_edges": int(m) if "m" in locals() else None,
            "exp_degree": exp_degree,
            "instance_id": instance_id,
            "time_s": float(time.perf_counter() - overall_t0),
            "event_idx": int(event_idx),
            "objective": int(objective) if objective is not None else None,
            "best_bound": None,
            "gap": None,
            "iteration": int(iteration) if iteration is not None else None,
            "note": note,
        })
        event_idx += 1

    rng = random.Random(seed)

    # ------------------------------------------------------------------
    # Read edges, build adjacency
    # ------------------------------------------------------------------
    edges = []
    max_node = -1
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            u, v = map(int, line.split())
            if u == v:
                continue
            edges.append((u, v))
            if u > max_node:
                max_node = u
            if v > max_node:
                max_node = v

    n = max_node + 1 if max_node >= 0 else 0
    m = len(edges)

    if n == 0:
        result = {
            "objective": 0,
            "best_bound": None,
            "wall_time": 0.0,
            "status": "Feasible",
            "embed_time": 0.0,
            "num_physical_qubits": 0.0,
            "max_chain_length": 0,
            "avg_chain_length": 0.0,
            "chain_strength": float(chain_strength),
            "subproblem_size": 0,
            "iters_used": 0,
            "n_nodes": 0,
            "n_edges": 0,
            "tile_pool_size": 0,
            "boundary_scale": 0.0,
        }
        log_progress(0, iteration=0, note="final")
        append_progress_rows(progress_csv_path, progress_rows)
        return result

    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)

    avg_deg = (2.0 * m / n) if n else 0.0

    # ------------------------------------------------------------------
    # Active nodes only
    # ------------------------------------------------------------------
    active_nodes = [u for u in range(n) if len(adj[u]) > 0]
    n_active = len(active_nodes)

    if n_active == 0:
        result = {
            "objective": 0,
            "best_bound": None,
            "wall_time": 0.0,
            "status": "Feasible",
            "embed_time": 0.0,
            "num_physical_qubits": 0.0,
            "max_chain_length": 0,
            "avg_chain_length": 0.0,
            "chain_strength": float(chain_strength),
            "subproblem_size": 0,
            "iters_used": 0,
            "n_nodes": int(n),
            "n_edges": int(m),
            "tile_pool_size": 0,
            "boundary_scale": 0.0,
        }
        log_progress(0, iteration=0, note="final")
        append_progress_rows(progress_csv_path, progress_rows)
        return result

    active_node_set = set(active_nodes)

    # ------------------------------------------------------------------
    # Improvement 1: Adaptive tile sizing
    # ------------------------------------------------------------------
    if avg_deg > 1.0:
        adaptive_size = int(2.0 * max_internal_edges / avg_deg)
        effective_subproblem_size = max(
            min_subproblem_size,
            min(subproblem_size, adaptive_size)
        )
    else:
        effective_subproblem_size = subproblem_size

    effective_subproblem_size = min(effective_subproblem_size, n_active)
    if n_active > 1:
        effective_subproblem_size = max(2, effective_subproblem_size)
    else:
        effective_subproblem_size = 1

    if verbose:
        print(
            f"[init] n={n} m={m} avg_deg={avg_deg:.2f}  "
            f"requested subproblem_size={subproblem_size}  "
            f"effective_subproblem_size={effective_subproblem_size}  "
            f"active_nodes={n_active}"
        )

    global_boundary_scale = boundary_scale_sparse if avg_deg <= 2.5 else boundary_scale_dense

    if pool_refresh_every is None:
        pool_refresh_every = max(reuse_cycles, 1)

    visit_count = [0] * n

    target = dnx.pegasus_graph(pegasus_m)
    target_edges = list(target.edges)
    sampler = RotorModelAnnealingSampler()

    if init_method == "zeros":
        x = [0] * n
    elif init_method == "random":
        x = [rng.randint(0, 1) for _ in range(n)]
    else:
        raise ValueError("init_method must be 'zeros' or 'random'")

    def compute_boundary_scale(Sset):
        n_internal = 0
        n_boundary = 0
        for u in Sset:
            for v in adj[u]:
                if v in Sset:
                    if u < v:
                        n_internal += 1
                else:
                    n_boundary += 1
        ratio = n_boundary / max(1, n_internal)
        t = min(ratio / 4.0, 1.0)
        return boundary_scale_min + t * (boundary_scale_max - boundary_scale_min)

    def build_tile_bqm(S, Sset, bs):
        lin = {}
        quad = {}

        for u in Sset:
            lin[u] = 0.0

        for u in Sset:
            for v in adj[u]:
                if v in Sset:
                    if u < v:
                        lin[u] -= 1.0
                        lin[v] -= 1.0
                        quad[(u, v)] = quad.get((u, v), 0.0) + 2.0
                else:
                    lin[u] += bs * (-1.0 + 2.0 * x[v])

        return dimod.BinaryQuadraticModel(lin, quad, 0.0, dimod.BINARY)

    def tile_unique_edges(Sset):
        es = []
        seen = set()
        for u in Sset:
            for v in adj[u]:
                a, b = (u, v) if u < v else (v, u)
                if (a, b) not in seen:
                    seen.add((a, b))
                    es.append((a, b))
        return es

    def delta_cut_for_edges(edge_list, old_vals, cand_vals):
        d = 0
        for u, v in edge_list:
            ou = old_vals.get(u, x[u])
            ov = old_vals.get(v, x[v])
            nu = cand_vals.get(u, ou)
            nv = cand_vals.get(v, ov)
            d += (1 if nu != nv else 0) - (1 if ou != ov else 0)
        return d

    def compute_hot_nodes(x_cur, sample_size=None):
        if m == 0:
            return []

        if sample_size is None:
            sample_size = min(m, max(1000, int(m * hot_edge_fraction * 10)))

        candidate_edges = edges
        if len(edges) > sample_size:
            indices = rng.sample(range(m), sample_size)
            candidate_edges = [edges[i] for i in indices]

        scored = []
        for u, v in candidate_edges:
            if x_cur[u] == x_cur[v]:
                cut_nbrs = sum(1 for w in adj[u] if x_cur[w] != x_cur[u])
                cut_nbrs += sum(1 for w in adj[v] if x_cur[w] != x_cur[v])
                scored.append((cut_nbrs, u, v))

        if not scored:
            return []

        scored.sort(reverse=True)
        top_k = max(1, int(len(scored) * hot_edge_fraction))
        hot_nodes = set()
        for _, u, v in scored[:top_k]:
            if u in active_node_set:
                hot_nodes.add(u)
            if v in active_node_set:
                hot_nodes.add(v)
        return list(hot_nodes)

    def weighted_seed_node(hot_nodes=None):
        hot_set = set(hot_nodes) if hot_nodes else set()
        candidates = active_nodes
        weights = []

        for u in candidates:
            w = 1.0 / (1.0 + visit_count[u] * coverage_bias)
            if u in hot_set:
                w *= cut_bias
            weights.append(w)

        total = sum(weights)
        if total <= 0:
            return rng.choice(candidates)

        r = rng.random() * total
        cumulative = 0.0
        for u, w in zip(candidates, weights):
            cumulative += w
            if cumulative >= r:
                return u
        return candidates[-1]

    def pick_tile(k, hot_nodes=None, max_restarts=50):
        S = set()
        q = deque()

        def seed_from_node(u):
            if u not in active_node_set:
                return
            if u not in S:
                S.add(u)
                q.append(u)

            if adj[u] and len(S) < k:
                nbrs = [v for v in adj[u] if v in active_node_set]
                if nbrs:
                    v = nbrs[rng.randrange(len(nbrs))]
                    if v not in S:
                        S.add(v)
                        q.append(v)

        if avg_deg >= 1.0:
            seed_from_node(weighted_seed_node(hot_nodes))
        else:
            seed_from_node(weighted_seed_node(hot_nodes))
            if len(S) < k and n_active > 1:
                seed_from_node(weighted_seed_node(hot_nodes))

        restarts = 0
        while len(S) < k:
            while q and len(S) < k:
                u = q.popleft()
                for v in adj[u]:
                    if v not in active_node_set:
                        continue
                    if v not in S:
                        S.add(v)
                        q.append(v)
                        if len(S) >= k:
                            break
            if len(S) >= k:
                break
            restarts += 1
            if restarts > max_restarts:
                break
            seed_from_node(weighted_seed_node(hot_nodes))

        return tuple(S)

    def try_embed_tile(S, attempt_seed):
        Sset = set(S)

        n_internal_edges = 0
        for u in Sset:
            for v in adj[u]:
                if v in Sset and u < v:
                    n_internal_edges += 1
        if n_internal_edges == 0:
            return None

        bs = compute_boundary_scale(Sset)
        bqm0 = build_tile_bqm(S, Sset, bs)
        if len(bqm0.quadratic) == 0:
            return None

        vars_int = list(bqm0.variables)
        to_str = {v: f"v{v}" for v in vars_int}
        to_int = {f"v{v}": v for v in vars_int}
        bqm_str = bqm0.relabel_variables(to_str, inplace=False)
        source_edges = list(bqm_str.quadratic)

        if len(source_edges) == 0:
            return None

        t0 = time.perf_counter()
        emb = find_embedding(source_edges, target_edges, random_seed=attempt_seed)
        elapsed = time.perf_counter() - t0

        if not emb:
            return None

        if set(bqm_str.variables) - set(emb.keys()):
            return None

        chain_lengths = [len(chain) for chain in emb.values()]
        num_physical = sum(chain_lengths)
        max_chain = max(chain_lengths) if chain_lengths else 0
        avg_chain = (num_physical / len(chain_lengths)) if chain_lengths else 0.0
        elist = tile_unique_edges(Sset)

        return (Sset, elist, emb, to_int, bs, num_physical, max_chain, avg_chain, elapsed)

    def solve_tile(tile_bqm, embedding, to_int, bs, local_seed):
        to_str = {v: f"v{v}" for v in tile_bqm.variables}
        bqm_str = tile_bqm.relabel_variables(to_str, inplace=False)

        missing = set(bqm_str.variables) - set(embedding.keys())
        if missing:
            return None, None

        embedded_bqm = embed_bqm(bqm_str, embedding, target, chain_strength=chain_strength)

        sample_kwargs = dict(
            num_reads=num_reads,
            num_sweeps=num_sweeps,
            beta_range=beta_range,
            beta_schedule_type=beta_schedule_type,
            Hp_field=Hp_field,
            Hd_field=Hd_field,
            randomize_order=randomize_order,
            trans_fields=trans_fields,
            num_sweeps_per_beta=num_sweeps_per_beta,
            proposal_acceptance_criteria=proposal_acceptance_criteria,
            seed=local_seed,
        )

        if warm_start:
            try:
                init_physical = {}
                for var_str, chain in embedding.items():
                    var_int = to_int.get(var_str)
                    val = x[var_int] if var_int is not None else 0
                    for qubit in chain:
                        init_physical[qubit] = val
                sample_kwargs["initial_states"] = [init_physical]
            except Exception:
                pass

        ss = sampler.sample(embedded_bqm, **sample_kwargs)

        timing = ss.info.get("timing", {})
        runtime = (sum(timing.values()) / 1e9) if timing else None

        if chain_break_method == "majority_vote":
            cbm = majority_vote
        elif chain_break_method == "discard":
            cbm = discard
        else:
            cbm = MinimizeEnergy(bqm_str, embedding)

        logical = unembed_sampleset(ss, embedding, bqm_str, chain_break_method=cbm)

        try:
            s = dict(logical.first.sample)
        except Exception:
            return None, runtime

        sample_int = {to_int[k]: int(v) for k, v in s.items() if k in to_int}
        return sample_int, runtime

    def refresh_pool(tile_pool, tile_meta, tile_improve_count,
                     n_retire, embed_time_acc, embed_seed_offset, hot_nodes):
        if not tile_pool or n_retire <= 0:
            return embed_time_acc, embed_seed_offset

        scored = sorted(tile_pool, key=lambda S: tile_improve_count.get(S, 0))
        to_retire = set(scored[:n_retire])

        new_pool = [S for S in tile_pool if S not in to_retire]
        for S in to_retire:
            tile_meta.pop(S, None)
            tile_improve_count.pop(S, None)

        new_pool_set = set(new_pool)
        attempts = 0
        while len(new_pool) < tile_pool_size and attempts < tile_pool_size * 6:
            attempts += 1
            embed_seed_offset += 1
            S = pick_tile(effective_subproblem_size, hot_nodes=hot_nodes)

            if len(S) < 2:
                continue
            if S in tile_meta or S in new_pool_set:
                continue

            result = try_embed_tile(S, seed + embed_seed_offset)
            if result is None:
                continue

            Sset, elist, emb, to_int_t, bs, num_physical, max_chain, avg_chain, elapsed = result
            embed_time_acc += elapsed
            tile_meta[S] = (Sset, elist, emb, to_int_t, bs, num_physical, max_chain, avg_chain)
            new_pool.append(S)
            new_pool_set.add(S)
            tile_improve_count[S] = tile_improve_count.get(S, 0)

            for u in Sset:
                visit_count[u] += 1

        tile_pool.clear()
        tile_pool.extend(new_pool)

        if verbose:
            print(f"  [refresh] retired={n_retire} pool_size={len(tile_pool)}")

        return embed_time_acc, embed_seed_offset

    embed_time = 0.0
    embed_seed_offset = 0
    tile_pool = []
    tile_meta = {}
    tile_improve_count = {}
    total_physical_qubits = 0
    max_chain_overall = 0
    avg_chain_accumulator = 0.0

    hot_nodes_cache = compute_hot_nodes(x)

    max_initial_attempts = max(tile_pool_size * 8, 32)
    for _ in range(max_initial_attempts):
        if len(tile_pool) >= tile_pool_size:
            break

        S = pick_tile(effective_subproblem_size, hot_nodes=hot_nodes_cache)
        if len(S) < 2:
            continue
        if S in tile_meta:
            continue

        embed_seed_offset += 1
        result = try_embed_tile(S, seed + embed_seed_offset)
        if result is None:
            continue

        Sset, elist, emb, to_int_t, bs, num_physical, max_chain, avg_chain, elapsed = result
        embed_time += elapsed

        total_physical_qubits += num_physical
        max_chain_overall = max(max_chain_overall, max_chain)
        avg_chain_accumulator += avg_chain

        tile_meta[S] = (Sset, elist, emb, to_int_t, bs, num_physical, max_chain, avg_chain)
        tile_pool.append(S)
        tile_improve_count[S] = 0

        for u in Sset:
            visit_count[u] += 1

    if tile_pool:
        avg_physical_qubits = total_physical_qubits / len(tile_pool)
        avg_chain_length = avg_chain_accumulator / len(tile_pool)
    else:
        avg_physical_qubits = 0.0
        avg_chain_length = 0.0

    cur_cut = sum(1 for u, v in edges if x[u] != x[v])
    best_cut = cur_cut

    log_progress(best_cut, iteration=0, note="initial")

    total_runtime = 0.0
    no_improve = 0
    iters_used = 0

    if not tile_pool:
        final_cut = sum(1 for u, v in edges if x[u] != x[v])
        result = {
            "objective": int(final_cut),
            "best_bound": None,
            "wall_time": float(total_runtime),
            "status": "NoEmbedding",
            "embed_time": float(embed_time),
            "num_physical_qubits": float(avg_physical_qubits),
            "max_chain_length": int(max_chain_overall),
            "avg_chain_length": float(avg_chain_length),
            "chain_strength": float(chain_strength),
            "subproblem_size": int(effective_subproblem_size),
            "iters_used": 0,
            "n_nodes": int(n),
            "n_edges": int(m),
            "tile_pool_size": 0,
            "boundary_scale": float(global_boundary_scale),
        }
        log_progress(int(final_cut), iteration=0, note="final")
        append_progress_rows(progress_csv_path, progress_rows)
        return result

    iter_budget = max_iters
    n_retire = max(1, int(len(tile_pool) * refresh_fraction))

    hot_nodes_cache = compute_hot_nodes(x)

    for it in range(iter_budget):
        iters_used = it + 1

        if not tile_pool:
            break

        if it > 0 and (it % pool_refresh_every == 0):
            hot_nodes_cache = compute_hot_nodes(x)
            embed_time, embed_seed_offset = refresh_pool(
                tile_pool, tile_meta, tile_improve_count,
                n_retire, embed_time, embed_seed_offset, hot_nodes_cache
            )
            if not tile_pool:
                break

        S = tile_pool[it % len(tile_pool)]
        Sset, elist, emb, to_int_t, bs, *_ = tile_meta[S]

        old_vals = {u: x[u] for u in Sset}
        tile_bqm = build_tile_bqm(S, Sset, bs)

        cand_sample, rt = solve_tile(tile_bqm, emb, to_int_t, bs, local_seed=seed + it + 1)

        if cand_sample is None:
            if S in tile_pool:
                tile_pool.remove(S)
            tile_meta.pop(S, None)
            tile_improve_count.pop(S, None)
            no_improve += 1
            if not tile_pool:
                break
            continue

        if rt is not None:
            total_runtime += rt

        cand_vals = {u: int(val) for u, val in cand_sample.items() if u in Sset}
        dcut = delta_cut_for_edges(elist, old_vals, cand_vals)

        accept = (dcut > 0) or (accept_worse_prob > 0.0 and rng.random() < accept_worse_prob)

        if accept:
            for u, val in cand_vals.items():
                x[u] = val
            cur_cut += dcut

            if dcut > 0:
                tile_improve_count[S] = tile_improve_count.get(S, 0) + dcut

            if cur_cut > best_cut:
                best_cut = cur_cut
                no_improve = 0
                log_progress(best_cut, iteration=it + 1, note="improved")
            else:
                no_improve += 0
        else:
            no_improve += 1

        for u in Sset:
            visit_count[u] += 1

        if log_checkpoints and checkpoint_every > 0 and ((it + 1) % checkpoint_every == 0):
            log_progress(best_cut, iteration=it + 1, note="checkpoint")

        if verbose:
            if it % max(1, (max_iters // 10)) == 0:
                print(
                    f"[iter {it}/{iter_budget}] cut={cur_cut} best={best_cut} "
                    f"no_improve={no_improve} pool={len(tile_pool)}"
                )

        if no_improve >= patience:
            break

    final_cut = sum(1 for u, v in edges if x[u] != x[v])
    objective = max(final_cut, best_cut)
    if objective > m:
        objective = m

    status = "Feasible"
    if iters_used == 0 and len(tile_pool) == 0:
        status = "NoEmbedding"

    result = {
        "objective": int(objective),
        "best_bound": None,
        "wall_time": float(total_runtime),
        "status": status,
        "embed_time": float(embed_time),
        "num_physical_qubits": float(avg_physical_qubits),
        "max_chain_length": int(max_chain_overall),
        "avg_chain_length": float(avg_chain_length),
        "chain_strength": float(chain_strength),
        "subproblem_size": int(effective_subproblem_size),
        "iters_used": int(iters_used),
        "n_nodes": int(n),
        "n_edges": int(m),
        "tile_pool_size": int(len(tile_pool)),
        "boundary_scale": float(global_boundary_scale),
    }

    log_progress(int(objective), iteration=iters_used, note="final")
    append_progress_rows(progress_csv_path, progress_rows)

    return result

## Wrapper


In [6]:
def SQA_auto(filename, verbose=True, **override_kwargs):
    """
    Wrapper around SQA_solve_hardware_like_v2 that automatically selects
    decomposition / search hyperparameters based on graph size and density.
    """

    edge_count = 0
    max_node = -1
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            u, v = map(int, line.split())
            if u == v:
                continue
            edge_count += 1
            if u > max_node:
                max_node = u
            if v > max_node:
                max_node = v

    n = max_node + 1
    m = edge_count
    avg_deg = (2.0 * m / n)
    sparse = avg_deg <= 3.0

    if verbose:
        print(f"[auto] n={n}  m={m}  avg_deg={avg_deg:.2f}  "
              f"regime={'sparse' if sparse else 'dense'}")

    if n <= 100:
        if sparse:
            regime = "tiny_sparse"
            params = dict(
                subproblem_size=100,
                tile_pool_size=24,
                reuse_cycles=10,
                max_iters=250,
                patience=25,
                pool_refresh_every=50,
                refresh_fraction=0.18,
                coverage_bias=1.2,
                cut_bias=2,
                warm_start=True,
                boundary_scale_sparse=0.35,
                boundary_scale_dense=0.70,
                boundary_scale_min=0.15,
                boundary_scale_max=1.20,
                max_internal_edges=1000,
                min_subproblem_size=60,
            )
        else:
            regime = "tiny_dense"
            params = dict(
                subproblem_size=60,
                tile_pool_size=36,
                reuse_cycles=45,
                max_iters=500,
                patience=50,
                pool_refresh_every=50,
                refresh_fraction=0.20,
                coverage_bias=1.5,
                cut_bias=1.6,
                warm_start=True,
                boundary_scale_sparse=0.50,
                boundary_scale_dense=1.00,
                boundary_scale_min=0.30,
                boundary_scale_max=1.50,
                max_internal_edges=1000,
                min_subproblem_size=60,
            )

    elif n <= 1_000:
        if sparse:
            regime = "medium_sparse"
            params = dict(
                subproblem_size=100,
                tile_pool_size=24,
                reuse_cycles=40,
                max_iters=500,
                patience=50,
                pool_refresh_every=40,
                refresh_fraction=0.20,
                coverage_bias=1.2,
                cut_bias=2,
                warm_start=True,
                boundary_scale_sparse=0.40,
                boundary_scale_dense=0.85,
                boundary_scale_min=0.20,
                boundary_scale_max=1.30,
                max_internal_edges=2200,
                min_subproblem_size=100,
            )
        else:
            regime = "medium_dense"
            params = dict(
                subproblem_size=80,
                tile_pool_size=36,
                reuse_cycles=40,
                max_iters=1500,
                patience=100,
                pool_refresh_every=40,
                refresh_fraction=0.25,
                coverage_bias=1.5,
                cut_bias=2,
                warm_start=True,
                boundary_scale_sparse=0.50,
                boundary_scale_dense=1.00,
                boundary_scale_min=0.25,
                boundary_scale_max=1.50,
                max_internal_edges=1000,
                min_subproblem_size=35,
            )

    elif n <= 10_000:
        if sparse:
            regime = "large_sparse"
            params = dict(
                subproblem_size=120,
                tile_pool_size=90,
                reuse_cycles=24,
                max_iters=1000,
                patience=100,
                pool_refresh_every=30,
                refresh_fraction=0.3,
                coverage_bias=3.0,
                cut_bias=2.0,
                warm_start=True,
                boundary_scale_sparse=0.40,
                boundary_scale_dense=0.85,
                boundary_scale_min=0.20,
                boundary_scale_max=1.30,
                max_internal_edges=1800,
                min_subproblem_size=120,
            )
        else:
            regime = "large_dense"
            params = dict(
                subproblem_size=60,
                tile_pool_size=128,
                reuse_cycles=48,
                max_iters=1500,
                patience=150,
                pool_refresh_every=30,
                refresh_fraction=0.30,
                coverage_bias=2.5,
                cut_bias=2.5,
                warm_start=True,
                boundary_scale_sparse=0.55,
                boundary_scale_dense=1.10,
                boundary_scale_min=0.30,
                boundary_scale_max=1.50,
                max_internal_edges=1000,
                min_subproblem_size=30,
                chain_strength=1.9,
            )

    elif n <= 100_000:
        if sparse:
            regime = "huge_sparse"
            params = dict(
                subproblem_size=60,
                tile_pool_size=64,
                reuse_cycles=100,
                max_iters=5000,
                patience=100,
                pool_refresh_every=40,
                refresh_fraction=0.4,
                coverage_bias=3.5,
                cut_bias=2.0,
                warm_start=True,
                boundary_scale_sparse=0.40,
                boundary_scale_dense=0.80,
                boundary_scale_min=0.15,
                boundary_scale_max=1.20,
                max_internal_edges=2000,
                min_subproblem_size=60,
            )
        else:
            regime = "huge_dense"
            params = dict(
                subproblem_size=40,
                tile_pool_size=96,
                reuse_cycles=300,
                max_iters=5000,
                patience=100,
                pool_refresh_every=50,
                refresh_fraction=0.4,
                coverage_bias=4,
                cut_bias=2.5,
                warm_start=True,
                boundary_scale_sparse=0.60,
                boundary_scale_dense=1.20,
                boundary_scale_min=0.35,
                boundary_scale_max=1.50,
                max_internal_edges=800,
                min_subproblem_size=30,
                chain_strength=2.5,
            )

    else:
            regime = "mega"
            params = dict(
                subproblem_size=40,
                tile_pool_size=48,
                reuse_cycles=100,
                max_iters=5000,
                patience=100,
                pool_refresh_every=80,
                refresh_fraction=0.35,
                coverage_bias=3.0,
                cut_bias=1.5,
                warm_start=True,
                boundary_scale_sparse=0.40,
                boundary_scale_dense=0.80,
                boundary_scale_min=0.15,
                boundary_scale_max=1.20,
                max_internal_edges=1200,
                min_subproblem_size=20,
            )

    if verbose:
        print(f"[auto] selected regime: {regime}")
        print(f"[auto] params: {params}")

    params.update(override_kwargs)

    return SQA_solve_hardware_like_v2(filename, verbose=verbose, **params)

# **Reference method - Google OR-Tools CP-SAT solver**

In order to solve using Google OR's constraint programming model we need to convert the instance into Constraint Programming - Satisfiability (CP-SAT) model. It combines the modeling capabilities of Constraint Programming (CP) with the efficiency of modern Boolean Satisfiability (SAT) solvers.

Decision variables: one per node same as before
$$ x_i \in \{0,1\} \enspace ∀i ∈ V$$

For each edge in E, introduce a boolean indicating whether the edge is cut:
$$y_{ij} \in \{0,1\} $$
Which we want to equal 1 when $x_i \not= x_j$
i.e
$$y_{ij} = x_i ⊕x_j$$
Now our objective turns to:
$$\max \sum_{i,j \in E}{y_{ij}} $$

To enfore this we need to add linear constraints:
$$y_{ij} \le x_i + x_j$$
$$y_{ij} \le 2 - (x_i+x_j)$$
$$y_{ij} \ge x_i - x_j  $$
$$y_{ij} \ge x_j - x_i $$
these enfore the XOR logic

for each problem instance there are $n + m$ binary variables now, with $4 \times m$ constraints


In [7]:
def solve_maxcut_cpsat(
    filepath: str,
    time_limit_s: float = 60.0,
    num_workers: int = 1,
):
    """
    Solve unweighted Max-Cut using OR-Tools CP-SAT.

    Assumptions:
      - Edge list file has NO header.
      - Each line is: "u v" with node ids 1..n (1-indexed).
      - Filename format: "[n]_[exp degree]_[instance].txt" (we parse n from the start).

    Returns a dict with status/objective/etc.
    """
    # Read edges
    edges = []
    max_node = -1

    with open(filepath, "r") as f:
        for line in f:
            u, v = map(int, line.split())
            if u == v:
                continue
            edges.append((u, v))
            max_node = max(max_node, u, v)

    n = max_node + 1

    if not edges:
        raise ValueError("No edges found in file.")

    model = cp_model.CpModel()

    # Node side variables
    x = [model.new_bool_var(f"x_{i}") for i in range(n)]

    # Symmetry breaking: flip symmetry x -> 1-x
    #model.add(x[0] == 0)

    # Edge cut variables and constraints
    y = []
    for (u, v) in edges:
        cut = model.new_bool_var(f"y_{u}_{v}")

        # y = |x_u - x_v| for booleans
        model.add(cut <= x[u] + x[v])
        model.add(cut <= 2 - (x[u] + x[v]))
        model.add(cut >= x[u] - x[v])
        model.add(cut >= x[v] - x[u])

        y.append(cut)

    model.maximize(sum(y))

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = float(time_limit_s)
    solver.parameters.num_search_workers = int(num_workers)

    status = solver.solve(model)

    status_map = {
        cp_model.OPTIMAL: "OPTIMAL",
        cp_model.FEASIBLE: "FEASIBLE",
        cp_model.INFEASIBLE: "INFEASIBLE",
        cp_model.MODEL_INVALID: "MODEL_INVALID",
        cp_model.UNKNOWN: "UNKNOWN",
    }

    result = {
        "objective": int(round(solver.objective_value)),
        "best_bound": float(solver.best_objective_bound),
        "wall_time": float(solver.wall_time),
        "status": status_map.get(status, str(status)),
        "n_edges": len(edges)
    }

    return result


# **Reference method - Gurobi Optimiser**
Gurobi uses a MIQP solver to solve Max-Cut instances

In [8]:
def solve_maxcut_gurobi(
    instance_path,
    time_limit_s=60,
    threads=2,
    mip_gap=None,
    quiet=False,

    # ---- progress logging ----
    progress_csv_path=None,
    exp_degree=None,
    instance_id=None,
):
    """
    Local Gurobi MIQP Max-Cut solver for unweighted graphs.

    Returns a dict suitable for batch aggregation.
    """

    progress_rows = []
    event_idx = 0

    # ------------------
    # Read edge list
    # ------------------
    edges = []
    max_node = -1

    with open(instance_path, "r") as f:
        for line in f:
            u, v = map(int, line.split())
            if u == v:
                continue
            edges.append((u, v))
            max_node = max(max_node, u, v)

    n = max_node + 1 if max_node >= 0 else 0
    n_edges = len(edges)

    # ------------------
    # Build model
    # ------------------
    env_params = {
        "WLSACCESSID": ID,
        "WLSSECRET":   SECRET,
        "LICENSEID":   LISCENSE,
    }

    if quiet:
        env_params["OutputFlag"] = 0

    env = gp.Env(params=env_params)
    model = gp.Model("maxcut", env=env)

    model.Params.TimeLimit = time_limit_s
    model.Params.Threads = threads
    if mip_gap is not None:
        model.Params.MIPGap = mip_gap

    # Heuristics-first tuning
    model.Params.MIPFocus = 1
    model.Params.Heuristics = 0.9
    model.Params.NoRelHeurTime = 30
    model.Params.RINS = 2

    # Binary variables: side of cut
    x = model.addVars(n, vtype=GRB.BINARY)

    # Objective: sum (x_i + x_j - 2 x_i x_j)
    obj = gp.QuadExpr()
    for i, j in edges:
        obj.add(x[i])
        obj.add(x[j])
        obj.add(-2.0 * x[i] * x[j])

    model.setObjective(obj, GRB.MAXIMIZE)

    def log_row(time_s, objective, best_bound, gap, note):
        nonlocal event_idx
        if progress_csv_path is None:
            return
        progress_rows.append({
            "solver": "GUROBI",
            "instance_path": instance_path,
            "n_nodes": int(n),
            "n_edges": int(n_edges),
            "exp_degree": exp_degree,
            "instance_id": instance_id,
            "time_s": float(time_s) if time_s is not None else None,
            "event_idx": int(event_idx),
            "objective": float(objective) if objective is not None else None,
            "best_bound": float(best_bound) if best_bound is not None else None,
            "gap": float(gap) if gap is not None else None,
            "iteration": None,
            "note": note,
        })
        event_idx += 1

    def callback(model, where):
        if where == GRB.Callback.MIPSOL:
            incumbent = model.cbGet(GRB.Callback.MIPSOL_OBJ)
            best_bound = model.cbGet(GRB.Callback.MIPSOL_OBJBND)
            runtime = model.cbGet(GRB.Callback.RUNTIME)

            if incumbent is not None and abs(incumbent) > 1e-12:
                gap = abs(best_bound - incumbent) / abs(incumbent)
            elif incumbent is not None and best_bound is not None:
                gap = 0.0 if abs(best_bound - incumbent) <= 1e-12 else None
            else:
                gap = None

            log_row(runtime, incumbent, best_bound, gap, "new_incumbent")

    # ------------------
    # Solve
    # ------------------
    model.optimize(callback)

    # ------------------
    # Results
    # ------------------
    status = model.Status
    has_sol = model.SolCount > 0

    GUROBI_STATUS_MAP = {
        GRB.OPTIMAL: "OPTIMAL",
        GRB.TIME_LIMIT: "TIME_LIMIT",
        GRB.INTERRUPTED: "INTERRUPTED",
        GRB.INFEASIBLE: "INFEASIBLE",
    }

    final_obj = int(round(model.ObjVal)) if has_sol else None
    final_bound = float(model.ObjBound) if has_sol else None
    final_gap = float(model.MIPGap) if has_sol else None
    final_runtime = float(model.Runtime)

    result = {
        "objective": final_obj,
        "best_bound": final_bound,
        "wall_time": final_runtime,
        "status": GUROBI_STATUS_MAP.get(status, str(status)),
        "n_edges": n_edges,
    }

    log_row(final_runtime, final_obj, final_bound, final_gap, "final")
    append_progress_rows(progress_csv_path, progress_rows)

    model.dispose()
    env.dispose()

    return result

# Batch Solver

In [9]:
def normalize_method_key(method: str) -> str:
    method_key = method.strip().upper()
    if method_key in ("CPSAT", "CP_SAT"):
        method_key = "CP-SAT"
    return method_key

def result_row_key(row):
    try:
        return (
            str(row["solver"]).strip().upper(),
            int(float(row["nodes"])),
            float(row["exp_degree"]),
            int(float(row["inst_id"])),
        )
    except Exception:
        return None

def load_done_keys(results_csv):
    done = set()
    if not os.path.exists(results_csv):
        return done

    with open(results_csv, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            k = result_row_key(row)
            if k is not None:
                done.add(k)
    return done

def read_all_result_rows(results_csv):
    if not os.path.exists(results_csv):
        return []
    with open(results_csv, "r", newline="") as f:
        return list(csv.DictReader(f))

def write_all_result_rows(results_csv, fieldnames, rows):
    with open(results_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def append_or_replace_result_row(results_csv, fieldnames, row, replace=False):
    """
    If replace=False: append row normally.
    If replace=True: replace existing row with same (solver, nodes, exp_degree, inst_id),
    otherwise append if not found.
    """
    if not replace:
        append_result_row(results_csv, fieldnames, row)
        return

    target_key = result_row_key(row)
    rows = read_all_result_rows(results_csv)

    replaced = False
    for i, old_row in enumerate(rows):
        if result_row_key(old_row) == target_key:
            rows[i] = row
            replaced = True
            break

    if not replaced:
        rows.append(row)

    write_all_result_rows(results_csv, fieldnames, rows)

In [10]:
def run_batch(
    node_size: int,
    method: str,
    time_limit_s: float = 60.0,
    workers: int = 1,
    skip_done: bool = True,
    max_instances: int | None = None,
    stop_on_error: bool = False,
    fieldnames: list[str] | None = None,
    progress_csv_path: str | None = None,
):

    if fieldnames is None:
        fieldnames = [
            "solver",
            "nodes",
            "exp_degree",
            "edges",
            "inst_id",
            "objective",
            "time",
            "bound",
            "gap",
            "timelimit",
            "status",
            "embed_time",
            "iters_used"
        ]

    ensure_results_csv(RESULTS_CSV, fieldnames)

    method_key = normalize_method_key(method)
    files = collect_instance_files(DATASET_DIR, node_size)

    if method_key == "SA":
        solver_fn = SA_solve
        solver_call_kwargs = {}

    elif method_key == "SQA":
        solver_fn = SQA_auto
        solver_call_kwargs = {"verbose": False}

    elif method_key == "CP-SAT":
        solver_fn = solve_maxcut_cpsat
        solver_call_kwargs = {
            "time_limit_s": time_limit_s,
            "num_workers": workers
        }

    elif method_key == "GUROBI":
        solver_fn = solve_maxcut_gurobi
        solver_call_kwargs = {
            "time_limit_s": time_limit_s,
            "threads": workers,
            "quiet": True,
        }

    else:
        raise ValueError("method must be one of: SA, SQA, CP-SAT, GUROBI")

    done = load_done_keys(RESULTS_CSV)

    # ------------------------------------------------------------
    # Build candidate list FIRST.
    # This is the key change that makes "next 20 unfinished" work.
    # ------------------------------------------------------------
    candidate_files = []
    skipped = 0

    for (d, inst, path) in files:
        key = (method_key, int(node_size), float(d), int(inst))

        if skip_done and key in done:
            skipped += 1
            continue

        candidate_files.append((d, inst, path))

    # Only after filtering unfinished runs do we apply max_instances
    if max_instances is not None:
        candidate_files = candidate_files[:max_instances]

    ran = 0
    errors = 0

    for (d, inst, path) in candidate_files:
        print(
            f"[Run start] solver={method_key}, "
            f"n={node_size}, deg={d}, inst={inst}, "
            f"tl={time_limit_s}s"
        )

        try:
            call_kwargs = dict(solver_call_kwargs)

            # pass progress logging metadata only to solvers that support it
            if method_key in ("SQA", "GUROBI"):
                call_kwargs.update({
                    "progress_csv_path": progress_csv_path,
                    "exp_degree": float(d),
                    "instance_id": int(inst),
                })

            solution = solver_fn(path, **call_kwargs)

            objective = solution.get("objective", None)
            wall_time = float(solution.get("wall_time", 0.0))
            status = str(solution.get("status", "UNKNOWN"))

            best_bound = solution.get("best_bound", None)
            embed_time = solution.get("embed_time", None)
            iters_used = solution.get("iters_used", None)
            edges = solution.get("n_edges", None)

            if objective is not None:
                objective = int(objective)

            gap = None
            if (
                objective is not None
                and best_bound is not None
                and best_bound != 0
            ):
                gap = abs(objective - best_bound) / abs(best_bound)

            row = {
                "solver": method_key,
                "nodes": int(node_size),
                "exp_degree": float(d),
                "edges": edges,
                "inst_id": int(inst),
                "objective": objective,
                "time": wall_time,
                "bound": best_bound,
                "gap": gap,
                "timelimit": time_limit_s,
                "status": status,
                "embed_time": embed_time,
                "iters_used": iters_used,
            }

            # ------------------------------------------------------------
            # If skip_done=False, overwrite matching existing row instead
            # of appending a duplicate.
            # ------------------------------------------------------------
            append_or_replace_result_row(
                RESULTS_CSV,
                fieldnames,
                row,
                replace=(not skip_done),
            )

            ran += 1
            done.add((method_key, int(node_size), float(d), int(inst)))

        except Exception as e:
            errors += 1
            err_msg = f"{type(e).__name__}: {e}"

            row = {
                "solver": method_key,
                "nodes": int(node_size),
                "exp_degree": float(d),
                "edges": None,
                "inst_id": int(inst),
                "objective": -1,
                "time": 0.0,
                "bound": None,
                "gap": None,
                "timelimit": time_limit_s,
                "status": f"ERROR: {err_msg[:180]}",
                "embed_time": None,
                "iters_used": None,
            }

            append_or_replace_result_row(
                RESULTS_CSV,
                fieldnames,
                row,
                replace=(not skip_done),
            )

            print(
                f"[Run error] solver={method_key}, "
                f"n={node_size}, deg={d}, inst={inst} -> {err_msg}"
            )

            traceback.print_exc(limit=2)

            if stop_on_error:
                raise

    print(
        f"[Batch done] solver={method_key}, n={node_size} | "
        f"ran={ran}, skipped={skipped}, errors={errors}"
    )

    print(f"CSV: {RESULTS_CSV}")
    if progress_csv_path:
        print(f"Progress CSV: {progress_csv_path}")

# Solution of all 25 subsections

## **Solving 100 sized instances**

In [ ]:
run_batch(100, 'SA', time_limit_s=None)

In [ ]:
run_batch(100, 'CP-SAT', time_limit_s=10, workers=8)

In [ ]:
run_batch(100, 'GUROBI', time_limit_s = 10, progress_csv_path=PROGRESS_CSV)

In [ ]:
run_batch(100, 'SQA', time_limit_s=None, progress_csv_path=PROGRESS_CSV,  max_instances=20, skip_done=True)

## **Solving 1,000 sized instances**

In [ ]:
run_batch(1000, 'SA', time_limit_s=None)

In [ ]:
run_batch(1000, 'SQA', time_limit_s=None,progress_csv_path=PROGRESS_CSV,  max_instances=20, skip_done=True)

In [ ]:
run_batch(1000, 'CP-SAT', time_limit_s=30, workers=8, max_instances=20)

In [ ]:
run_batch(1000, 'GUROBI', time_limit_s = 30, max_instances=50, progress_csv_path=PROGRESS_CSV, skip_done=True)

## **Solving 10,000 sized instances**

In [ ]:
run_batch(10_000, 'SA', time_limit_s=None)

In [ ]:
run_batch(10_000, 'SQA', time_limit_s=None, progress_csv_path=PROGRESS_CSV,  max_instances=2, skip_done=True)

In [ ]:
run_batch(10_000, 'CP-SAT', time_limit_s=60, workers=8)

In [ ]:
run_batch(10_000, 'GUROBI', time_limit_s = 60, max_instances=20, progress_csv_path=PROGRESS_CSV, skip_done=True)

## **Solving 100,000 sized instances**

In [ ]:
run_batch(100_000, 'SA', time_limit_s=None)

In [ ]:
run_batch(100_000, 'SQA', time_limit_s=None, progress_csv_path=PROGRESS_CSV, skip_done=True)

In [ ]:
run_batch(100_000, 'CP-SAT', time_limit_s=180, workers=2, max_instances=1)

In [ ]:
run_batch(100_000, 'GUROBI', time_limit_s = 180, max_instances=50, progress_csv_path=PROGRESS_CSV, skip_done=True)

## **Solving 1,000,000 sized instances**

In [ ]:
run_batch(1_000_000, 'SA', time_limit_s=None, max_instances=100, skip_done=True)

In [ ]:
run_batch(1_000_000, 'SQA', time_limit_s=None, max_instances=1, progress_csv_path=PROGRESS_CSV, skip_done=True)

In [ ]:
# can only do 0.6 -> 3 (60 total) exceeds RAM
run_batch(1_000_000, 'CP-SAT', time_limit_s=300, workers=1, max_instances=10, skip_done=True)

In [ ]:
run_batch(1_000_000, 'GUROBI', time_limit_s=300, max_instances=100, progress_csv_path=PROGRESS_CSV, skip_done=True)